# CIHI Wait Times EDA

**CIND 820 Big Data Analytics Project, Milestone 2: Architecture and Data Audit**

Mamkon Mercy Oyeleke | Student Number: 501279489 | Toronto Metropolitan University

**Data source:** Canadian Institute for Health Information - Wait Times for Priority Procedures in Canada, 2025 Data Tables
(https://www.cihi.ca/en/wait-times-for-priority-procedures-in-canada-2025)

**Outputs:** `../outputs/figures/fig8` through `fig10`, and `../outputs/tables/cihi_ontario_2024_summary.csv`

Run the setup cell below first. It clones this repository when running in Google Colab, so the relative `../data/raw/` and `../outputs/` paths used throughout this notebook resolve correctly.

## Setup

In [ ]:
import os, sys

# If running in Google Colab, clone the repo so relative data/output paths resolve correctly
if 'google.colab' in sys.modules:
    if not os.path.exists('Architecture_and_Data_Audit'):
        os.system('git clone https://github.com/mamkon/Architecture_and_Data_Audit.git')
    os.chdir('Architecture_and_Data_Audit/notebooks')

print("Working directory:", os.getcwd())

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150,
})
PALETTE = ['#185FA5', '#1D9E75', '#D85A30', '#BA7517', '#993556', '#639922', '#534AB7', '#5F5E5A']

## Section 1: LOAD AND PROFILE

In [ ]:
print("=" * 70)
print("SECTION 1 - LOADING AND PROFILING CIHI WAIT TIMES DATA")
print("=" * 70)

FILE = '../data/raw/CIHI_Wait_Times_Priority_Procedures_2025.xlsx'
df = pd.read_excel(FILE, sheet_name='Table 1', header=1)
df = df[df['Province'].notna()].copy()
df = df[df['Reporting level'].isin(['Provincial', 'National', 'Regional'])]

print(f"\nTotal rows (provincial/national/regional): {len(df):,}")
print(f"Columns: {list(df.columns[:8])}")
print(f"Provinces: {df['Province'].nunique()}")
print(f"Indicators: {df['Indicator'].nunique()}")
print(f"Metrics: {df['Metric'].unique().tolist()}")
print(f"Missing 'Indicator result': {df['Indicator result'].isna().sum()} "
      f"({df['Indicator result'].isna().mean()*100:.1f}%)")

## Section 2: ONTARIO PROVINCIAL TRENDS, 2014-2024

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2 - ONTARIO PROVINCIAL TRENDS (50th PERCENTILE)")
print("=" * 70)

on = df[(df['Province'] == 'Ontario') & (df['Reporting level'] == 'Provincial')].copy()
on_num = on[on['Data year'].apply(lambda x: isinstance(x, (int, float)))]
on_num = on_num[(on_num['Metric'] == '50th Percentile') & (on_num['Data year'] >= 2014)]

key_inds = ['Hip Replacement', 'Knee Replacement', 'Cataract Surgery', 'CT Scan', 'MRI Scan']

fig, ax = plt.subplots(figsize=(11, 6))
for i, ind in enumerate(key_inds):
    sub = on_num[on_num['Indicator'] == ind].sort_values('Data year')
    ax.plot(sub['Data year'], sub['Indicator result'], marker='o', linewidth=2,
            color=PALETTE[i], label=ind)
ax.set_xlabel('Year')
ax.set_ylabel('Median (50th percentile) wait time, days')
ax.set_title('Ontario Median Wait Times by Procedure, 2014-2024')
ax.legend(fontsize=9)
ax.axvspan(2019.5, 2021.5, alpha=0.08, color='gray')
ax.text(2020.2, ax.get_ylim()[1] * 0.05, 'COVID-19', fontsize=8, color='gray', ha='center')
plt.tight_layout()
plt.savefig('../outputs/figures/fig8_cihi_ontario_trends.png', bbox_inches='tight')
plt.close()
print("Saved fig8_cihi_ontario_trends.png")

print("\nObservation: all five procedures show a sharp spike in wait times during")
print("2020-2021, consistent with COVID-19 surgical backlogs. By 2024, wait times")
print("for hip/knee replacement and cataract surgery had partially recovered but")
print("remained above 2019 levels, while MRI wait times continued climbing.")

## Section 3: ONTARIO REGIONAL BREAKDOWN (2024)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3 - ONTARIO REGIONAL WAIT TIMES (2024)")
print("=" * 70)

on_reg = df[(df['Province'] == 'Ontario') & (df['Reporting level'] == 'Regional')].copy()
print(f"\nOntario regional rows: {len(on_reg)}")
print(f"Regions: {sorted(on_reg['Region'].dropna().unique().tolist())}")
print(f"Indicators with regional granularity: {sorted(on_reg['Indicator'].dropna().unique().tolist())}")
print(f"Years available: {sorted([y for y in on_reg['Data year'].dropna().unique() if isinstance(y,(int,float))])}")

on_reg_2024 = on_reg[(on_reg['Data year'] == 2024) & (on_reg['Metric'] == '50th Percentile')]
piv2024 = on_reg_2024.pivot_table(index='Region', columns='Indicator', values='Indicator result')
print("\n2024 median wait (days), Hip/Knee Replacement, by Ontario region:")
print(piv2024.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
piv2024.plot(kind='barh', ax=ax, color=['#185FA5', '#1D9E75'], edgecolor='white')
ax.set_xlabel('Median wait time (days)')
ax.set_title('2024 Median Wait Times for Hip & Knee Replacement by Ontario Region')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/fig9_cihi_ontario_regions_2024.png', bbox_inches='tight')
plt.close()
print("\nSaved fig9_cihi_ontario_regions_2024.png")

print("\nKEY STRUCTURAL FINDING:")
print("CIHI's 'Regional' granularity for Ontario applies to only 2 of 14")
print("indicators (Hip Replacement, Knee Replacement) and uses 6 broad health")
print("regions (Central, East, North East, North West, Toronto, West) - not")
print("PHO's 37 public health unit geographies. This is the binding constraint")
print("on RQ3's spatial resolution: any burden-vs-access analysis can only be")
print("conducted at the 6-region level, not the 37-PHU level.")

## Section 4: PROVINCIAL BENCHMARK COMPARISON (2024)

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4 - % MEETING BENCHMARK, ALL PROVINCES (2024)")
print("=" * 70)

bench = df[(df['Reporting level'] == 'Provincial') & (df['Metric'] == '% Meeting Benchmark')
           & (df['Data year'] == 2024)]
bench_piv = bench.pivot_table(index='Province', columns='Indicator', values='Indicator result')
print("\n2024 % Meeting Benchmark, all provinces:")
print(bench_piv.round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 7))
bench_piv[['Hip Replacement', 'Knee Replacement', 'Cataract Surgery']].plot(
    kind='barh', ax=ax, color=['#185FA5', '#1D9E75', '#D85A30'], edgecolor='white')
ax.axvline(50, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('% of patients meeting wait time benchmark')
ax.set_title('2024 Benchmark Performance by Province')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/fig10_cihi_benchmark_by_province.png', bbox_inches='tight')
plt.close()
print("\nSaved fig10_cihi_benchmark_by_province.png")

on_2024 = on[on['Data year'] == 2024]
on_2024[['Indicator', 'Metric', 'Unit of measurement', 'Indicator result']].to_csv(
    '../outputs/tables/cihi_ontario_2024_summary.csv', index=False)
print("Saved cihi_ontario_2024_summary.csv")

print("\nOntario sits mid-pack on hip/knee replacement benchmarks (82.0% / 79.2%),")
print("ahead of Quebec and the Atlantic provinces but behind Alberta on hip")
print("replacement specifically. Ontario leads on radiation therapy (97.7%).")

print("\n" + "=" * 70)
print("CIHI EDA COMPLETE - see 03_pho_cihi_join_rq3.py for the regional join")
print("=" * 70)